# 02 — Data Quality & Cleaning

Audits the 51-ticker daily OHLCV dataset produced by Task 2, applies
`clean_ohlcv()`, and produces a structured quality report covering:

1. **Missing values** — three distinct categories (pre-listing, within-life, calendar misalignment)
2. **Outlier detection** — `adj_close`-based return outliers (±50%) and volume spikes (>5× trailing-20d avg)
3. **Survivorship bias** — documented caveat on the universe construction

Goal: make every data problem **visible and documented**, not silently patched.

In [1]:
import logging
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.universe import END_DATE, START_DATE, UNIVERSE
from src.data_utils import clean_ohlcv, load_ohlcv

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
matplotlib.use("Agg")
CACHE = str(repo_root / "data" / "raw")

print(f"Universe: {len(UNIVERSE)} tickers | {START_DATE} → {END_DATE}")

Universe: 51 tickers | 2020-08-30 → 2025-08-30


## 0. Load data and confirm Task 2 fix is in place

In [2]:
daily = load_ohlcv(UNIVERSE, START_DATE, END_DATE, cache_dir=CACHE)

obs = daily.groupby("ticker").size()
modal = int(obs.mode()[0])
short = obs[obs < modal]

print(f"Total rows: {len(daily):,}  |  Tickers: {daily.ticker.nunique()}")
print(f"Modal obs/ticker (full 5-yr): {modal}")
print()
if not short.empty:
    print("Tickers with fewer than full coverage (pre-listing dropna fix active):")
    for t, n in short.items():
        first = daily[daily.ticker == t]["date"].min().date()
        print(f"  {t}: {n} obs, first trade {first}, missing {modal - n} pre-IPO days")
else:
    print("All tickers have full 5-yr coverage.")

# Confirm zero pre-IPO phantom rows for PLTR
pltr_pre = daily[(daily.ticker == "PLTR") & (daily.date < "2020-09-30")]
assert len(pltr_pre) == 0, f"Found {len(pltr_pre)} pre-IPO PLTR rows — fix not in place!"
print("\nTask 2 fix confirmed: zero pre-IPO phantom rows for PLTR.")

INFO Loading OHLCV from cache: /Users/johnnyli/Desktop/Lendo Capital/data/raw/ohlcv_daily.parquet


Total rows: 64,035  |  Tickers: 51
Modal obs/ticker (full 5-yr): 1256

Tickers with fewer than full coverage (pre-listing dropna fix active):
  PLTR: 1235 obs, first trade 2020-09-30, missing 21 pre-IPO days

Task 2 fix confirmed: zero pre-IPO phantom rows for PLTR.


## 1. Run `clean_ohlcv()` — produces cleaned data + full quality report

In [3]:
cleaned, report = clean_ohlcv(daily)

print(f"cleaned shape: {cleaned.shape}  (input: {daily.shape})")
print()
print("Quality report summary:")
for key, df in report.items():
    print(f"  {key:25s}: {len(df):4d} rows")

INFO No within-life gaps to fill — data is complete.


INFO 84 volume spike event(s) across 26 ticker(s) (threshold: 5× trailing-20d avg).


cleaned shape: (64035, 8)  (input: (64035, 8))

Quality report summary:
  pre_listing_gaps         :    0 rows
  within_life_gaps         :    0 rows
  partial_nans             :    0 rows
  calendar_misalign        :    0 rows
  filled_rows              :    0 rows
  return_outliers          :    0 rows
  volume_spikes            :   84 rows


## 2. Missing-value analysis — three categories

Missing values have different root causes and require different responses.
Lumping them together hides the distinction.

| Category | Cause | Expected? | Action |
|---|---|---|---|
| **Pre-listing** | Ticker didn't exist yet | Yes | Drop (already done in `load_ohlcv`) |
| **Within-life gaps** | Missing date in market calendar after first trade | No (large-caps) | Flag; ffill ≤2d |
| **Calendar misalignment** | Multiple established tickers missing same date | No | Flag; investigate |

In [4]:
# --- Category 1: Pre-listing gaps ---
pre = report["pre_listing_gaps"]
print("=== Category 1: Pre-listing gaps ===")
if pre.empty:
    print("  None — load_ohlcv() already dropped all pre-listing rows (Task 2 fix).")
    print("  PLTR is the only post-2020-08-31 IPO; its pre-listing phantom rows were")
    print("  removed in the Task 2 dropna fix (21 rows, dates 2020-08-31 to 2020-09-29).")
else:
    print(pre.to_string())

=== Category 1: Pre-listing gaps ===
  None — load_ohlcv() already dropped all pre-listing rows (Task 2 fix).
  PLTR is the only post-2020-08-31 IPO; its pre-listing phantom rows were
  removed in the Task 2 dropna fix (21 rows, dates 2020-08-31 to 2020-09-29).


In [5]:
# --- Category 2: Within-life gaps ---
wlg = report["within_life_gaps"]
print("=== Category 2: Within-life gaps (dates missing after first trade) ===")
if wlg.empty:
    print("  None — all 51 tickers have a complete trading-day record after their")
    print("  first trade date. Expected for large-cap US equities with no delistings.")
else:
    print(f"  {len(wlg)} gap(s) across {wlg.ticker.nunique()} ticker(s):")
    print(wlg.to_string(index=False))

=== Category 2: Within-life gaps (dates missing after first trade) ===
  None — all 51 tickers have a complete trading-day record after their
  first trade date. Expected for large-cap US equities with no delistings.


In [6]:
# --- Category 3: Partial NaN in existing rows ---
pn = report["partial_nans"]
print("=== Category 3a: Partial NaN in existing rows ===")
if pn.empty:
    print("  None — no row has a NaN in any price or volume column.")
else:
    print(pn.to_string(index=False))

print()

# --- Calendar misalignment ---
cm = report["calendar_misalign"]
print("=== Category 3b: Calendar misalignment (multiple established tickers missing same date) ===")
if cm.empty:
    print("  None — no date has multiple established tickers simultaneously absent.")
    print("  Confirms the batch yfinance download used a consistent US market calendar.")
else:
    print(cm.to_string(index=False))

=== Category 3a: Partial NaN in existing rows ===
  None — no row has a NaN in any price or volume column.

=== Category 3b: Calendar misalignment (multiple established tickers missing same date) ===
  None — no date has multiple established tickers simultaneously absent.
  Confirms the batch yfinance download used a consistent US market calendar.


In [7]:
# --- Fill log ---
fr = report["filled_rows"]
print("=== Forward-fill log ===")
if fr.empty:
    print("  Nothing was filled — no within-life gaps existed in the data.")
    print("  The cleaned DataFrame is identical to the input for all 51 tickers.")
else:
    print(f"  {len(fr)} row(s) forward-filled across {fr.ticker.nunique()} ticker(s):")
    print(fr.to_string(index=False))

print()
print("Fill policy applied (see clean_ohlcv docstring for full rationale):")
print("  · Forward-fill price columns only, limit=2 trading days")
print("  · Volume → NaN on filled rows (no defensible carry-forward for volume)")
print("  · Never fill before first_trade_date")
print("  · Gaps > 2 days left as NaN and flagged")

=== Forward-fill log ===
  Nothing was filled — no within-life gaps existed in the data.
  The cleaned DataFrame is identical to the input for all 51 tickers.

Fill policy applied (see clean_ohlcv docstring for full rationale):
  · Forward-fill price columns only, limit=2 trading days
  · Volume → NaN on filled rows (no defensible carry-forward for volume)
  · Never fill before first_trade_date
  · Gaps > 2 days left as NaN and flagged


## 3. % Missing by year

Even though the missing-value counts above are zero, computing % missing by year
per column is still worth doing — it provides a baseline check and is the right
structure to re-run if the universe changes or data is refreshed.

In [8]:
price_vol = ["open", "high", "low", "close", "adj_close", "volume"]
daily_copy = daily.copy()
daily_copy["year"] = daily_copy["date"].dt.year

pct_missing = (
    daily_copy.groupby("year")[price_vol]
    .apply(lambda g: g.isna().mean() * 100)
    .round(4)
)
print("% missing by year (all columns):")
print(pct_missing.to_string())

# Row-level % missing
total_cells = len(daily) * len(price_vol)
null_cells  = daily[price_vol].isna().sum().sum()
print(f"\nOverall: {null_cells} null cells out of {total_cells:,} ({null_cells/total_cells*100:.4f}%)")

% missing by year (all columns):
      open  high  low  close  adj_close  volume
year                                           
2020   0.0   0.0  0.0    0.0        0.0     0.0
2021   0.0   0.0  0.0    0.0        0.0     0.0
2022   0.0   0.0  0.0    0.0        0.0     0.0
2023   0.0   0.0  0.0    0.0        0.0     0.0
2024   0.0   0.0  0.0    0.0        0.0     0.0
2025   0.0   0.0  0.0    0.0        0.0     0.0

Overall: 0 null cells out of 384,210 (0.0000%)


## 4. Ticker-level coverage report — IPO confirmation

In [9]:
coverage = (
    daily.groupby("ticker")["date"]
    .agg(first_trade="min", last_trade="max", n_days="count")
    .sort_values("first_trade", ascending=False)
)
coverage["days_vs_full"] = modal - coverage["n_days"]

print("Tickers with first_trade after window start (2020-08-31):")
late = coverage[coverage["first_trade"] > pd.Timestamp("2020-08-31")]
if late.empty:
    print("  None — all tickers except PLTR were trading before the window start.")
    # PLTR has first_trade = 2020-09-30 which IS > 2020-08-31
else:
    print(late.to_string())
    print()
    print("Public IPO/listing confirmation:")
    print("  PLTR (Palantir): direct listing on NYSE on 2020-09-30. ✓")

print()
print("Full coverage table (sorted by first_trade descending):")
coverage

Tickers with first_trade after window start (2020-08-31):
       first_trade last_trade  n_days  days_vs_full
ticker                                             
PLTR    2020-09-30 2025-08-29    1235            21

Public IPO/listing confirmation:
  PLTR (Palantir): direct listing on NYSE on 2020-09-30. ✓

Full coverage table (sorted by first_trade descending):


,first_trade,last_trade,n_days,days_vs_full
ticker,,,,
PLTR,2020-09-30,2025-08-29,1235,21
AAPL,2020-08-31,2025-08-29,1256,0
PM,2020-08-31,2025-08-29,1256,0
META,2020-08-31,2025-08-29,1256,0
MRK,2020-08-31,2025-08-29,1256,0
MSFT,2020-08-31,2025-08-29,1256,0
NFLX,2020-08-31,2025-08-29,1256,0
NOW,2020-08-31,2025-08-29,1256,0
NVDA,2020-08-31,2025-08-29,1256,0


## 5. Outlier detection

### 5a. `adj_close` daily return outliers (threshold: ±50%)

Uses `adj_close`, **not** raw `close`. Using raw close would false-flag NVDA's
10-for-1 split on 2024-06-10 as a −90% crash (raw close dropped from ~\$1,200
to ~\$120). With `adj_close` (already split-adjusted), the split date shows
a normal ~+0.7% return.

In [10]:
ro = report["return_outliers"]
print(f"adj_close return outliers (|return| > 50%): {len(ro)} events")
if ro.empty:
    print("  None found — the universe of 51 large-cap stocks had no single-day")
    print("  adj_close move exceeding ±50% in the 5-year window.")
else:
    print(ro.to_string(index=False))

print()

# Explicit NVDA split non-flag check
nvda_split_window = cleaned[
    (cleaned.ticker == "NVDA") &
    (cleaned.date.between("2024-06-07", "2024-06-13"))
].copy()
nvda_split_window["adj_ret"] = (
    nvda_split_window.set_index("date")["adj_close"].pct_change().values
)
print("NVDA adj_close around the 2024-06-10 (10-for-1) split:")
print(nvda_split_window[["date", "adj_close", "adj_ret"]].to_string(index=False))
print()
split_ret = nvda_split_window[nvda_split_window.date == pd.Timestamp("2024-06-10")]["adj_ret"].iloc[0]
assert abs(split_ret) < 0.50, f"NVDA split date wrongly flagged as outlier: {split_ret:.2%}"
print(f"NVDA split date return: {split_ret:.4%}  ← correctly NOT flagged as an outlier.")

adj_close return outliers (|return| > 50%): 0 events
  None found — the universe of 51 large-cap stocks had no single-day
  adj_close move exceeding ±50% in the 5-year window.

NVDA adj_close around the 2024-06-10 (10-for-1) split:
      date  adj_close   adj_ret
2024-06-07 120.679176       NaN
2024-06-10 121.579605  0.007461
2024-06-11 120.711052 -0.007144
2024-06-12 124.993973  0.035481
2024-06-13 129.396759  0.035224

NVDA split date return: 0.7461%  ← correctly NOT flagged as an outlier.


### 5b. Volume spikes (threshold: >5× trailing-20-day average)

**Threshold rationale:** 5× is statistically extreme for large-cap equities
(>4σ on a lognormal volume distribution). At this level, spikes almost always
coincide with genuine market events — earnings surprises, index rebalances,
M&A announcements, quarterly option expirations. They are flagged rather than
removed: a volume spike is real market information, not a data error.

In [11]:
vs = report["volume_spikes"].copy()
vs["vol_ratio"] = vs["vol_ratio"].round(1)
vs["vol_ma20"]  = vs["vol_ma20"].round(0)

print(f"Volume spikes (>5× trailing-20d avg): {len(vs)} events across {vs.ticker.nunique()} tickers")
print()
print("Top 20 by spike ratio:")
print(vs.head(20).to_string(index=False))

Volume spikes (>5× trailing-20d avg): 84 events across 26 tickers

Top 20 by spike ratio:
ticker       date       volume   vol_ma20  vol_ratio
  NFLX 2022-04-20 1333875000.0 52097250.0       25.6
  NFLX 2022-01-21  589043000.0 39526200.0       14.9
   LIN 2024-03-15   33347400.0  2319460.0       14.4
   CRM 2024-05-30   66763300.0  4740835.0       14.1
   RTX 2023-07-25   49625600.0  4032015.0       12.3
  UBER 2023-12-15  364261200.0 30938910.0       11.8
   LIN 2021-06-25   23412000.0  2075995.0       11.3
   LIN 2022-06-24   23958800.0  2408410.0        9.9
  ABBV 2021-09-01   50943200.0  5229480.0        9.7
   LLY 2024-10-30   18257200.0  2074715.0        8.8
  PLTR 2024-02-06  420501900.0 48006515.0        8.8
   UNH 2025-05-15  121849200.0 13940435.0        8.7
   IBM 2023-03-17   37400200.0  4406840.0        8.5
  PLTR 2024-09-20  450290500.0 53273835.0        8.5
  ORCL 2023-12-12   57666500.0  6838180.0        8.4
  PLTR 2021-08-12  189287200.0 22634740.0        8.4
  META 20

In [12]:
print("Volume spikes by ticker (count):")
print(
    vs.groupby("ticker").size()
    .sort_values(ascending=False)
    .rename("spike_events")
    .to_string()
)

Volume spikes by ticker (count):
ticker
NFLX    9
ORCL    8
IBM     7
PLTR    6
UNH     5
CRM     5
LIN     5
LLY     5
MRK     4
META    4
RTX     3
ABBV    3
WMT     3
NOW     2
QCOM    2
TMO     2
UBER    2
KO      1
ISRG    1
NVDA    1
GS      1
GE      1
AVGO    1
TXN     1
AMGN    1
AAPL    1


In [13]:
# Spot-check: NFLX 2022-04-20 — earnings miss (-35% price drop, panic selling)
nflx_row = vs[vs.ticker == "NFLX"].iloc[0]
print("Largest single spike — NFLX 2022-04-20:")
print(f"  Volume:      {nflx_row['volume']:>15,.0f} shares")
print(f"  20d avg:     {nflx_row['vol_ma20']:>15,.0f} shares")
print(f"  Spike ratio: {nflx_row['vol_ratio']:>15.1f}×")
print()
# Show price action that day
nflx_that_day = cleaned[
    (cleaned.ticker == "NFLX") &
    (cleaned.date.between("2022-04-18", "2022-04-22"))
][["date","open","high","low","adj_close","volume"]]
print("NFLX price action around spike:")
print(nflx_that_day.to_string(index=False))
print()
print("Context: NFLX reported its first subscriber loss in a decade on 2022-04-19.")
print("Stock fell ~35% the next day on 25× normal volume — a real event, not a data error.")

Largest single spike — NFLX 2022-04-20:
  Volume:        1,333,875,000 shares
  20d avg:          52,097,250 shares
  Spike ratio:            25.6×

NFLX price action around spike:
      date      open      high       low  adj_close       volume
2022-04-18 34.000000 34.236000 33.161999  33.785999   51050000.0
2022-04-19 33.321999 35.167999 33.321999  34.861000  209069000.0
2022-04-20 24.520000 24.870001 21.250999  22.618999 1333875000.0
2022-04-21 22.000000 22.768000 21.152000  21.822001  535016000.0
2022-04-22 22.018000 22.627001 21.004999  21.552000  375151000.0

Context: NFLX reported its first subscriber loss in a decade on 2022-04-19.
Stock fell ~35% the next day on 25× normal volume — a real event, not a data error.


In [14]:
# Volume spike timeline plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(vs["date"], vs["vol_ratio"], alpha=0.7, s=40, color="steelblue")
ax.axhline(5, color="red", linestyle="--", lw=1, label="5× threshold")
ax.set_title("Volume spikes across the universe (>5× trailing-20d avg)")
ax.set_xlabel("Date")
ax.set_ylabel("Volume ratio (× trailing avg)")
ax.legend()
# Annotate the top 5; push the highest point's label below to avoid title overlap
for i, (_, row) in enumerate(vs.head(5).iterrows()):
    ax.annotate(
        f"{row['ticker']}\n{row['vol_ratio']:.0f}×",
        xy=(row["date"], row["vol_ratio"]),
        xytext=(8, -14 if i == 0 else 4), textcoords="offset points",
        fontsize=7, va="top" if i == 0 else "bottom",
    )
plt.tight_layout()
plt.savefig("volume_spikes.png", dpi=100)
plt.show()
print("Chart saved to volume_spikes.png")

Chart saved to volume_spikes.png


/var/folders/35/d26c1c9s0cnbfnfsmk98p0sw0000gn/T/ipykernel_45970/1529665276.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Survivorship bias — explicit caveat

**This section must be read before interpreting any return or performance
statistic produced in Tasks 4–5.**

---

### What survivorship bias means in this dataset

The universe was constructed by taking the **top 50 S&P 500 constituents by
market cap as of 2025-08-30** and applying that snapshot retroactively across
the full 5-year window (2020-08-31 → 2025-08-29). This introduces a
structural bias in two directions:

**Companies excluded despite being top-50 historically:**  
Firms that were among the largest in the S&P 500 during 2020–2022 but have
since lost market cap — through business deterioration, M&A, or sector
rotation — are **absent** from this dataset, even for the years they belonged.
Examples include companies that were top-50 in 2020 but are no longer: some
former tech leaders, energy majors at their 2022 peaks, or healthcare
companies that have since de-rated.

**Companies included despite not being top-50 historically:**  
Firms that grew *into* the top 50 recently are included for their **entire**
window history, even the early period when they were a smaller company.
The clearest example in this dataset:

- **PLTR (Palantir):** IPO'd at \$10/share on 2020-09-30. By 2025 it had
  grown ~18× to ~\$180/share and entered the S&P 500 (September 2024),
  reaching top-50 market cap status. Its full post-IPO history is in this
  dataset, including the period 2020–2023 when it was far outside the top 50.
  An investor constructing a top-50 portfolio in 2020 would not have held PLTR.

### Net effect on Tasks 4–5 statistics

The bias is **upward** — systematically toward companies that succeeded:

- Excluded companies (the failures and underperformers) drag performance down
  in reality but contribute zero to this dataset.
- Included late-joiners like PLTR contribute their strong appreciation history,
  which would not have been available to a real investor running this strategy.

**Any average return or Sharpe ratio computed in Tasks 4–5 will be higher
than what an investor living through 2020–2025 without hindsight would have
achieved.** The magnitude of the bias is hard to quantify without point-in-time
historical index data, but for high-growth stocks like PLTR it can easily
account for several percentage points of annualised return.

### What is NOT done to correct this

A proper fix requires point-in-time S&P 500 constituent data (e.g., from
Compustat, Bloomberg, or CRSP) — out of scope for this week. The bias is
documented here so it is never forgotten or silently baked into a reported
statistic.

**Rule for Tasks 4–5:** any performance figure must carry the qualifier
*"using a survivorship-biased universe (top-50 S&P 500 as of 2025-08-30)"*.

In [15]:
# Illustrate survivorship bias: PLTR's adj_close over the full window
pltr = cleaned[cleaned.ticker == "PLTR"].copy()
pltr_ret_5yr = pltr.adj_close.iloc[-1] / pltr.adj_close.iloc[0] - 1

# Compare to SPY over the same (shorter) window
spy_from_pltr = cleaned[
    (cleaned.ticker == "SPY") & (cleaned.date >= pltr.date.min())
].copy()
spy_ret_same_window = spy_from_pltr.adj_close.iloc[-1] / spy_from_pltr.adj_close.iloc[0] - 1

print("PLTR vs SPY — same window (2020-09-30 to window end):")
print(f"  PLTR total return: {pltr_ret_5yr:+.1%}")
print(f"  SPY  total return: {spy_ret_same_window:+.1%}")
print()
print("PLTR's outperformance is real — but retroactively including a stock that")
print("grew 18× in a 'top-50 as of today' universe inflates the universe's")
print("historical average return well above what was achievable with foresight.")

# Simple illustration: PLTR weight in a hypothetical equal-weight top-50 portfolio
pltr_weight = 1 / 50
pltr_contribution = pltr_weight * pltr_ret_5yr
print(f"\nIn an equal-weight 50-stock portfolio, PLTR (2% weight) would have")
print(f"contributed {pltr_contribution:+.1%} to total return — purely from look-ahead.")

PLTR vs SPY — same window (2020-09-30 to window end):
  PLTR total return: +1549.6%
  SPY  total return: +106.0%

PLTR's outperformance is real — but retroactively including a stock that
grew 18× in a 'top-50 as of today' universe inflates the universe's
historical average return well above what was achievable with foresight.

In an equal-weight 50-stock portfolio, PLTR (2% weight) would have
contributed +31.0% to total return — purely from look-ahead.


## 7. Task 3 quality-check summary

In [16]:
qc = [
    ("Pre-listing gaps",   len(report['pre_listing_gaps']) == 0,
     "0 rows (all dropped by load_ohlcv Task-2 fix)"),
    ("Within-life gaps",   len(report['within_life_gaps']) == 0,
     "0 rows (data complete for all 51 tickers post first-trade)"),
    ("Partial NaN rows",   len(report['partial_nans']) == 0,
     "0 rows (no column-level NaN in any existing row)"),
    ("Calendar misalign",  len(report['calendar_misalign']) == 0,
     "0 events (consistent US market calendar across batch download)"),
    ("Fill log complete",  True,
     f"{len(report['filled_rows'])} rows filled (0 — nothing to fill)"),
    ("Return outliers",    len(report['return_outliers']) == 0,
     "0 events (no adj_close move > ±50% in 5yr window)"),
    ("NVDA split not flagged", abs(split_ret) < 0.50,
     f"NVDA 2024-06-10 adj_close return = {split_ret:.4%} (not flagged ✓)"),
    ("Volume spikes catalogued", len(report['volume_spikes']) > 0,
     f"{len(report['volume_spikes'])} events across {report['volume_spikes'].ticker.nunique()} tickers"),
    ("Survivorship bias documented", True,
     "Written up in Section 6 above"),
]

print(f"{'Check':<30} {'Pass':^6}  Detail")
print("-" * 80)
all_pass = True
for name, ok, detail in qc:
    mark = "✓" if ok else "✗"
    print(f"{name:<30} {mark:^6}  {detail}")
    if not ok:
        all_pass = False

print()
print("ALL QUALITY CHECKS PASSED — safe to proceed to Task 4 (Returns)." if all_pass
      else "SOME CHECKS FAILED — review above before proceeding.")

Check                           Pass   Detail
--------------------------------------------------------------------------------
Pre-listing gaps                 ✓     0 rows (all dropped by load_ohlcv Task-2 fix)
Within-life gaps                 ✓     0 rows (data complete for all 51 tickers post first-trade)
Partial NaN rows                 ✓     0 rows (no column-level NaN in any existing row)
Calendar misalign                ✓     0 events (consistent US market calendar across batch download)
Fill log complete                ✓     0 rows filled (0 — nothing to fill)
Return outliers                  ✓     0 events (no adj_close move > ±50% in 5yr window)
NVDA split not flagged           ✓     NVDA 2024-06-10 adj_close return = 0.7461% (not flagged ✓)
Volume spikes catalogued         ✓     84 events across 26 tickers
Survivorship bias documented     ✓     Written up in Section 6 above

ALL QUALITY CHECKS PASSED — safe to proceed to Task 4 (Returns).
